# 02 · Feature engineering (Home Credit Default Risk)

Builds one row per applicant by aggregating the history tables, saves a parquet file, then scores LightGBM on it.
Run the cells top to bottom. Expect about 10 to 15 minutes in total on a free Colab runtime.

## 1. Setup: where are the CSVs?

In [8]:
import os, gc
from pathlib import Path

# ---- EDIT THIS: where the Kaggle CSVs live -----------------------------------
# Option A (default): Google Drive folder. Put all Kaggle CSVs in this folder.
USE_DRIVE = False
DRIVE_RAW = "/content/drive/MyDrive/credit-risk/data/raw"
DRIVE_OUT = "/content/drive/MyDrive/credit-risk/data/processed"

# Option B: files uploaded straight into the Colab session (left panel, Files).
# Set USE_DRIVE = False and they are read from /content.
# -------------------------------------------------------------------------------

EXPECTED_ROWS = 307511   # rows in application_train.csv

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RAW, OUT = Path(DRIVE_RAW), Path(DRIVE_OUT)
else:
    RAW, OUT = Path("/content"), Path("/content")
OUT.mkdir(parents=True, exist_ok=True)

needed = ["application_train.csv", "bureau.csv", "previous_application.csv",
          "installments_payments.csv", "POS_CASH_balance.csv", "credit_card_balance.csv"]
missing = [f for f in needed if not (RAW / f).exists()]
assert not missing, f"Missing in {RAW}: {missing}"
print("All files found in", RAW)

All files found in /content


## 2. Helpers and application-level features

In [2]:
import numpy as np
import pandas as pd

ID_COL = "SK_ID_CURR"
PLACEHOLDER = 365243   # Home Credit's "not applicable" value in DAYS_* columns

def agg_table(df, group_col, aggs, prefix):
    """Group, aggregate, flatten column names to PREFIX_COLUMN_FUNC."""
    out = df.groupby(group_col).agg(aggs)
    out.columns = [f"{prefix}_{c}_{f}".upper() for c, f in out.columns]
    return out.reset_index()

def safe_div(a, b):
    """a / b with zero denominators turned into NaN (never inf)."""
    return a / b.replace(0, np.nan)

app = pd.read_csv(RAW / "application_train.csv")
assert app.shape[0] == EXPECTED_ROWS, f"app has {app.shape[0]} rows, expected {EXPECTED_ROWS}"
assert app[ID_COL].is_unique

app["DAYS_EMPLOYED_ANOM"] = (app["DAYS_EMPLOYED"] == PLACEHOLDER).astype(int)
app["DAYS_EMPLOYED"] = app["DAYS_EMPLOYED"].replace(PLACEHOLDER, np.nan)

app["CREDIT_INCOME_RATIO"]  = safe_div(app["AMT_CREDIT"], app["AMT_INCOME_TOTAL"])
app["ANNUITY_INCOME_RATIO"] = safe_div(app["AMT_ANNUITY"], app["AMT_INCOME_TOTAL"])
app["CREDIT_TERM"]          = safe_div(app["AMT_ANNUITY"], app["AMT_CREDIT"])
app["EMPLOYED_AGE_RATIO"]   = safe_div(app["DAYS_EMPLOYED"], app["DAYS_BIRTH"])
ext = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
app["EXT_MEAN"] = app[ext].mean(axis=1)
app["EXT_PROD"] = app[ext].prod(axis=1)
print("app:", app.shape)

app: (307511, 129)


## 3. History tables (each reduced to one row per applicant)
Only the needed columns are read, and each big table is freed after aggregation to keep Colab's RAM under control.

In [3]:
# --- bureau: credits at other institutions ---
bureau = pd.read_csv(RAW / "bureau.csv", usecols=[
    ID_COL, "SK_ID_BUREAU", "CREDIT_ACTIVE", "CREDIT_DAY_OVERDUE", "DAYS_CREDIT",
    "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE"])
bureau["ACTIVE"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)
bureau["OVERDUE_FLAG"] = (bureau["CREDIT_DAY_OVERDUE"] > 0).astype(int)
bureau_agg = agg_table(bureau, ID_COL, {
    "SK_ID_BUREAU": ["count"],
    "ACTIVE": ["mean", "sum"],
    "OVERDUE_FLAG": ["mean"],
    "AMT_CREDIT_SUM": ["sum", "mean", "max"],
    "AMT_CREDIT_SUM_DEBT": ["sum", "mean"],
    "AMT_CREDIT_SUM_OVERDUE": ["sum"],
    "DAYS_CREDIT": ["min", "max", "mean"],
}, "BUREAU")
bureau_agg["BUREAU_DEBT_CREDIT_RATIO"] = safe_div(
    bureau_agg["BUREAU_AMT_CREDIT_SUM_DEBT_SUM"], bureau_agg["BUREAU_AMT_CREDIT_SUM_SUM"])
del bureau; gc.collect()

# --- previous applications at Home Credit ---
day_cols = ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION",
            "DAYS_LAST_DUE", "DAYS_TERMINATION"]
prev = pd.read_csv(RAW / "previous_application.csv", usecols=[
    ID_COL, "SK_ID_PREV", "NAME_CONTRACT_STATUS", "AMT_ANNUITY", "AMT_APPLICATION",
    "AMT_CREDIT", "DAYS_DECISION", *day_cols])
for c in day_cols:
    prev[c] = prev[c].replace(PLACEHOLDER, np.nan)
prev["REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
prev["APP_CREDIT_RATIO"] = safe_div(prev["AMT_APPLICATION"], prev["AMT_CREDIT"])
prev_agg = agg_table(prev, ID_COL, {
    "SK_ID_PREV": ["count"],
    "REFUSED": ["mean", "sum"],
    "AMT_ANNUITY": ["mean", "max"],
    "AMT_CREDIT": ["sum", "mean"],
    "APP_CREDIT_RATIO": ["mean"],
    "DAYS_DECISION": ["min", "max"],
}, "PREV")
del prev; gc.collect()
print("bureau_agg:", bureau_agg.shape, "| prev_agg:", prev_agg.shape)

bureau_agg: (305811, 15) | prev_agg: (338857, 11)


In [4]:
# --- installment payment behaviour (about 13M rows, the heaviest table) ---
inst = pd.read_csv(RAW / "installments_payments.csv", usecols=[
    ID_COL, "SK_ID_PREV", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT", "AMT_PAYMENT"])
inst["DAYS_LATE"] = inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]
inst["LATE_FLAG"] = (inst["DAYS_LATE"] > 0).astype(int)
inst["UNDERPAID"] = (inst["AMT_PAYMENT"] < inst["AMT_INSTALMENT"] - 0.01).astype(int)
inst["PAY_RATIO"] = safe_div(inst["AMT_PAYMENT"], inst["AMT_INSTALMENT"])
inst_agg = agg_table(inst, ID_COL, {
    "SK_ID_PREV": ["nunique"],
    "DAYS_LATE": ["mean", "max"],
    "LATE_FLAG": ["mean", "sum"],
    "UNDERPAID": ["mean", "sum"],
    "PAY_RATIO": ["mean", "min"],
}, "INST")
del inst; gc.collect()

# --- POS / cash loan balances ---
pos = pd.read_csv(RAW / "POS_CASH_balance.csv", usecols=[
    ID_COL, "SK_ID_PREV", "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"])
pos_agg = agg_table(pos, ID_COL, {
    "SK_ID_PREV": ["nunique"],
    "MONTHS_BALANCE": ["min", "size"],
    "SK_DPD": ["max", "mean"],
    "SK_DPD_DEF": ["max", "mean"],
}, "POS")
del pos; gc.collect()

# --- credit card balances ---
cc = pd.read_csv(RAW / "credit_card_balance.csv", usecols=[
    ID_COL, "SK_ID_PREV", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_DRAWINGS_CURRENT", "SK_DPD"])
cc["UTILIZATION"] = safe_div(cc["AMT_BALANCE"], cc["AMT_CREDIT_LIMIT_ACTUAL"])
cc_agg = agg_table(cc, ID_COL, {
    "SK_ID_PREV": ["nunique"],
    "UTILIZATION": ["mean", "max"],
    "SK_DPD": ["max", "mean"],
    "AMT_BALANCE": ["mean", "max"],
    "AMT_DRAWINGS_CURRENT": ["sum", "mean"],
}, "CC")
del cc; gc.collect()
print("inst_agg:", inst_agg.shape, "| pos_agg:", pos_agg.shape, "| cc_agg:", cc_agg.shape)

inst_agg: (334834, 10) | pos_agg: (337252, 8) | cc_agg: (103558, 10)


## 4. Merge, check, save

In [5]:
full = app
for name, agg in [("bureau", bureau_agg), ("prev", prev_agg), ("inst", inst_agg),
                  ("pos", pos_agg), ("cc", cc_agg)]:
    assert agg[ID_COL].is_unique, f"{name}: duplicate {ID_COL}"
    full = full.merge(agg, on=ID_COL, how="left")
    assert len(full) == EXPECTED_ROWS, f"row count changed after merging {name}: {len(full)}"

# no inf values may survive
num_cols = full.select_dtypes("number").columns
full[num_cols] = full[num_cols].replace([np.inf, -np.inf], np.nan)

full.to_parquet(OUT / "train_features.parquet")
print("full:", full.shape, "->", OUT / "train_features.parquet")

full: (307511, 178) -> /content/train_features.parquet


## 5. Score LightGBM on the full feature set
Same split and CV settings as Phase 1 (`random_state=42`, 80/20 stratified, 5-fold). Phase 1 baseline: ROC-AUC 0.7569, PR-AUC 0.2402.

In [6]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from lightgbm import LGBMClassifier

X = full.drop(columns=["TARGET", ID_COL])
y = full["TARGET"]
for c in X.select_dtypes(include=["object", "string"]).columns:
    X[c] = X[c].astype("category")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print("train:", X_train.shape, "| test:", X_test.shape)

lgb = LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31,
                     subsample=0.8, colsample_bytree=0.7, random_state=42, verbose=-1)
cv = StratifiedKFold(5, shuffle=True, random_state=42)
r = cross_validate(lgb, X_train, y_train, cv=cv, scoring=["roc_auc", "average_precision"])
print("ROC-AUC", r["test_roc_auc"].mean().round(4),
      "| PR-AUC", r["test_average_precision"].mean().round(4),
      "| fit_sec", r["fit_time"].mean().round(1))

train: (246008, 176) | test: (61503, 176)
ROC-AUC 0.7788 | PR-AUC 0.2692 | fit_sec 60.4


## 6. Optional: which feature group earns its place?
Slow (five full CV runs, roughly 20 minutes). Leave `RUN_ABLATION = False` unless you want the table for your README.

In [7]:
RUN_ABLATION = False

if RUN_ABLATION:
    groups = {
        "ratios": ["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_TERM",
                   "EMPLOYED_AGE_RATIO", "EXT_MEAN", "EXT_PROD"],
        "bureau": [c for c in X_train.columns if c.startswith("BUREAU_")],
        "prev":   [c for c in X_train.columns if c.startswith("PREV_")],
        "inst":   [c for c in X_train.columns if c.startswith("INST_")],
        "pos":    [c for c in X_train.columns if c.startswith("POS_")],
        "cc":     [c for c in X_train.columns if c.startswith("CC_")],
    }
    used = {c for cols in groups.values() for c in cols}
    cols = [c for c in X_train.columns if c not in used]   # raw application columns
    rows = []
    for step, name in [(None, "raw application only")] + [(g, f"+ {g}") for g in groups]:
        if step:
            cols = cols + groups[step]
        r = cross_validate(lgb, X_train[cols], y_train, cv=cv,
                           scoring=["roc_auc", "average_precision"])
        rows.append({"step": name, "n_features": len(cols),
                     "roc_auc": round(float(r["test_roc_auc"].mean()), 4),
                     "pr_auc": round(float(r["test_average_precision"].mean()), 4)})
        print(rows[-1])
    print(pd.DataFrame(rows).to_string(index=False))